In [ ]:
import torch 
from PIL import Image as PIL_Image
from transformers import MllamaforConditionalGeneration, Autoprocessor, MllamaProcessor

model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"

model = MllamaforConditionalGeneration.from_pretrained(model_id, device_map = "auto", dtype = torch.bfloat16)
processor = MllamaProcessor.from_pretrained(model_id)

with open("Path_to_image", "rb") as f:
    raw_image = PIL_Image.open(f).convert("RGB")
conversation = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Describe this image in two sentances"},
        ],
    },
]

prompt = processor.apply_chat_template(
        conversation, add_generation_prompt = True, 
        tokenize=False,
        add_generation_prompt = True,
        tokenize = False,
        )

inputs = processor(
    prompt,
    raw_image,
    return tensors = "pt",
).to(model.device)

output = model.generate(
    **inputs,
    temperature = 0.7,
    top_p = 0.9,
    max_new_tokens = 512,
)

print("text&image_output = ", processor.decode(output[0])[len(prompt):])
